# SQL Analysis of a Sales Database (SQLite)

**Goal:** Design a small relational sales database, populate it with realistic sample data, and answer business questions using SQL.

**Tools:** SQLite, Python, Pandas, SQL (`JOIN`, `GROUP BY`, `WHERE`, `HAVING`, subqueries, CTEs, and window functions).

Run every cell from top to bottom in Google Colab. No external dataset or internet connection is required.

## 1. Import libraries and create the SQLite database

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display

# Create an in-memory SQLite database for this notebook.
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
print("SQLite database created successfully.")

## 2. Design the database schema

In [ ]:
schema_sql = """
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city TEXT NOT NULL,
    signup_date TEXT NOT NULL
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    unit_price REAL NOT NULL CHECK (unit_price >= 0)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price REAL NOT NULL CHECK (unit_price >= 0),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
"""

conn.executescript(schema_sql)
print("Tables created: customers, products, orders, order_items")

## 3. Insert sample business data

In [ ]:
customers_data = [
    (1, "Aarav Sharma", "Bengaluru", "2025-01-10"),
    (2, "Diya Patel", "Mumbai", "2025-01-15"),
    (3, "Rohan Verma", "Delhi", "2025-02-01"),
    (4, "Ananya Rao", "Bengaluru", "2025-02-12"),
    (5, "Kabir Singh", "Hyderabad", "2025-03-05"),
    (6, "Meera Nair", "Chennai", "2025-03-18"),
    (7, "Ishaan Gupta", "Pune", "2025-04-02"),
    (8, "Sara Khan", "Mumbai", "2025-04-20")
]

products_data = [
    (101, "Laptop", "Electronics", 65000),
    (102, "Wireless Mouse", "Accessories", 1200),
    (103, "Keyboard", "Accessories", 2500),
    (104, "Smartphone", "Electronics", 30000),
    (105, "Office Chair", "Furniture", 8500),
    (106, "Desk", "Furniture", 12000),
    (107, "Headphones", "Electronics", 4500),
    (108, "USB-C Hub", "Accessories", 2200)
]

orders_data = [
    (1001, 1, "2025-05-01", "Delivered"),
    (1002, 2, "2025-05-03", "Delivered"),
    (1003, 3, "2025-05-05", "Cancelled"),
    (1004, 1, "2025-05-12", "Delivered"),
    (1005, 4, "2025-05-15", "Delivered"),
    (1006, 5, "2025-06-02", "Shipped"),
    (1007, 6, "2025-06-08", "Delivered"),
    (1008, 2, "2025-06-11", "Delivered"),
    (1009, 7, "2025-06-18", "Pending"),
    (1010, 8, "2025-06-22", "Delivered"),
    (1011, 4, "2025-07-01", "Delivered"),
    (1012, 5, "2025-07-04", "Delivered")
]

order_items_data = [
    (1, 1001, 101, 1, 65000), (2, 1001, 102, 2, 1200),
    (3, 1002, 104, 1, 30000), (4, 1002, 107, 1, 4500),
    (5, 1003, 105, 1, 8500), (6, 1004, 103, 2, 2500),
    (7, 1004, 108, 1, 2200), (8, 1005, 106, 1, 12000),
    (9, 1005, 105, 2, 8500), (10, 1006, 101, 1, 65000),
    (11, 1006, 107, 2, 4500), (12, 1007, 102, 3, 1200),
    (13, 1007, 103, 1, 2500), (14, 1008, 104, 1, 30000),
    (15, 1008, 108, 2, 2200), (16, 1009, 105, 1, 8500),
    (17, 1010, 107, 1, 4500), (18, 1010, 102, 1, 1200),
    (19, 1011, 101, 1, 65000), (20, 1011, 106, 1, 12000),
    (21, 1012, 103, 2, 2500), (22, 1012, 108, 2, 2200)
]

conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customers_data)
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products_data)
conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders_data)
conn.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", order_items_data)
conn.commit()

for table in ["customers", "products", "orders", "order_items"]:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS rows_count FROM {table}", conn).iloc[0, 0]
    print(f"{table}: {count} rows")

## 4. Explore the tables

In [ ]:
for table in ["customers", "products", "orders", "order_items"]:
    print(f"\\n--- {table.upper()} ---")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5", conn))

## 5. Business Question 1 — What is the total revenue from delivered orders?

In [ ]:
query = """
SELECT ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Delivered';
"""
display(pd.read_sql_query(query, conn))

**Concepts:** `JOIN` combines orders with their line items; `WHERE` filters delivered orders; `SUM` calculates revenue.

## 6. Business Question 2 — Which products generate the most revenue?

In [ ]:
query = """
SELECT
    p.product_name,
    p.category,
    SUM(oi.quantity) AS units_sold,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'Delivered'
GROUP BY p.product_id, p.product_name, p.category
ORDER BY revenue DESC;
"""
display(pd.read_sql_query(query, conn))

## 7. Business Question 3 — How much revenue does each product category generate?

In [ ]:
query = """
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS category_revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'Delivered'
GROUP BY p.category
ORDER BY category_revenue DESC;
"""
display(pd.read_sql_query(query, conn))

## 8. Business Question 4 — Who are the top customers by spending?

In [ ]:
query = """
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent,
    COUNT(DISTINCT o.order_id) AS delivered_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Delivered'
GROUP BY c.customer_id, c.customer_name, c.city
ORDER BY total_spent DESC
LIMIT 5;
"""
display(pd.read_sql_query(query, conn))

## 9. Business Question 5 — Which cities have the highest sales?

In [ ]:
query = """
SELECT
    c.city,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS city_revenue,
    COUNT(DISTINCT o.order_id) AS delivered_orders
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Delivered'
GROUP BY c.city
ORDER BY city_revenue DESC;
"""
display(pd.read_sql_query(query, conn))

## 10. Business Question 6 — Which months have the highest revenue?

In [ ]:
query = """
SELECT
    strftime('%Y-%m', o.order_date) AS order_month,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS monthly_revenue,
    COUNT(DISTINCT o.order_id) AS delivered_orders
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Delivered'
GROUP BY strftime('%Y-%m', o.order_date)
ORDER BY order_month;
"""
monthly_revenue = pd.read_sql_query(query, conn)
display(monthly_revenue)

## 11. Business Question 7 — Which customers have placed more than one delivered order?

In [ ]:
query = """
SELECT
    c.customer_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Delivered'
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(DISTINCT o.order_id) > 1
ORDER BY order_count DESC, total_spent DESC;
"""
display(pd.read_sql_query(query, conn))

**Concepts:** `HAVING` filters aggregated groups, while `WHERE` filters rows before grouping.

## 12. Business Question 8 — What is the order status breakdown?

In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS number_of_orders
FROM orders
GROUP BY status
ORDER BY number_of_orders DESC;
"""
display(pd.read_sql_query(query, conn))

## 13. Business Question 9 — Find products priced above the average product price

In [ ]:
query = """
SELECT product_name, category, unit_price
FROM products
WHERE unit_price > (SELECT AVG(unit_price) FROM products)
ORDER BY unit_price DESC;
"""
display(pd.read_sql_query(query, conn))

## 14. Business Question 10 — Rank customers by delivered-order revenue

In [ ]:
query = """
WITH customer_sales AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'Delivered'
    GROUP BY c.customer_id, c.customer_name
)
SELECT
    customer_name,
    ROUND(revenue, 2) AS revenue,
    DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
FROM customer_sales
ORDER BY revenue_rank, customer_name;
"""
display(pd.read_sql_query(query, conn))

## 15. Create a reusable SQL query helper

In [ ]:
def run_query(sql):
    """Run a SQL query against the SQLite database and display its result."""
    result = pd.read_sql_query(sql, conn)
    display(result)
    return result

# Example: list all delivered orders worth more than 20,000.
run_query("""
SELECT
    o.order_id,
    o.order_date,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS order_total
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.status = 'Delivered'
GROUP BY o.order_id, o.order_date, c.customer_name
HAVING SUM(oi.quantity * oi.unit_price) > 20000
ORDER BY order_total DESC;
""")

## 16. Visualize monthly revenue

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(monthly_revenue["order_month"], monthly_revenue["monthly_revenue"])
plt.title("Monthly Delivered-Order Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 17. Export the database and query results

In [ ]:
# Save a portable SQLite database file.
backup_path = "sales_analysis.db"
backup_conn = sqlite3.connect(backup_path)
conn.backup(backup_conn)
backup_conn.close()

# Export selected business results to CSV.
product_sales = pd.read_sql_query("""
SELECT p.product_name,
       SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
JOIN orders o ON oi.order_id = o.order_id
WHERE o.status = 'Delivered'
GROUP BY p.product_id, p.product_name
ORDER BY revenue DESC;
""", conn)

product_sales.to_csv("product_sales_summary.csv", index=False)
print("Created sales_analysis.db and product_sales_summary.csv")

## 18. Practice exercises

Try writing the SQL yourself before checking the hints.

1. Find customers who have never placed an order. *(Hint: `LEFT JOIN` and `IS NULL`)*
2. Find the average value of a delivered order. *(Hint: calculate each order total in a subquery)*
3. Find the best-selling product by quantity. *(Hint: `SUM(quantity)` and `ORDER BY`)*
4. Find total revenue by order status. *(Hint: join `orders` and `order_items`, then `GROUP BY status`)*
5. Find customers whose delivered-order spending is above ₹50,000. *(Hint: `HAVING`)*

## Key SQL concepts covered

| SQL concept | Where used |
|---|---|
| `CREATE TABLE` | Database schema |
| `INSERT INTO` | Sample records |
| `SELECT`, `WHERE` | Filtering |
| `INNER JOIN` | Combining related tables |
| `LEFT JOIN` | Practice exercise |
| `GROUP BY` | Sales summaries |
| `HAVING` | Filtering grouped results |
| `ORDER BY`, `LIMIT` | Ranking and top results |
| Subquery | Average product price |
| CTE | Customer sales ranking |
| Window function | `DENSE_RANK()` |

## Conclusion

You designed a relational sales database and used SQL to answer practical business questions about revenue, products, customers, cities, monthly trends, and order statuses. The SQLite database and CSV summary are exported for reuse.